### **Instalasi Dependensi**

In [73]:
!pip install catboost cmaes xgboost lightgbm --quiet

# **Import Library**

In [177]:
import warnings
import time
import random
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize
from sklearn.impute import KNNImputer, SimpleImputer

import xgboost as xgb
from cmaes import CatCMAwM

In [75]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

In [76]:
DATA_PATH        = "./RTA Dataset.csv"
TARGET_COL       = "Accident_severity"
TEST_SIZE        = 0.15   # 20% untuk test set final
VAL_SIZE         = 0.15   # 20% dari train untuk validation set
K_WEAK_LEARNERS  = 5      # Jumlah weak learner per ensemble
N_ESTIMATORS_WL  = 200    # Pohon per weak learner (kecil agar cepat)
EVO_MAX_ITER     = 50     # Jumlah generasi CMA-ES evolutionary boosting
JOINT_MAX_ITER   = 30     # Jumlah generasi joint optimization
LAMBDA_PENALTY   = 0.05   # Koefisien penalti kompleksitas genome

# **Import Dataset**

In [77]:
df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,Service_year_of_vehicle,Defect_of_vehicle,Area_accident_occured,Lanes_or_Medians,Road_allignment,Types_of_Junction,Road_surface_type,Road_surface_conditions,Light_conditions,Weather_conditions,Type_of_collision,Number_of_vehicles_involved,Number_of_casualties,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Casualty_severity,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident,Accident_severity
0,17:02:00,Monday,18-30,Male,Above high school,Employee,1-2yr,Automobile,Owner,Above 10yr,No defect,Residential areas,NaN,Tangent road with flat terrain,No junction,Asphalt roads,Dry,Daylight,Normal,Collision with roadside-parked vehicles,2,2,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Moving Backward,Slight Injury
1,17:02:00,Monday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,5-10yrs,No defect,Office areas,Undivided Two way,Tangent road with flat terrain,No junction,Asphalt roads,Dry,Daylight,Normal,Vehicle with vehicle collision,2,2,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury
2,17:02:00,Monday,18-30,Male,Junior high school,Employee,1-2yr,Lorry (41?100Q),Owner,NaN,No defect,Recreational areas,other,NaN,No junction,Asphalt roads,Dry,Daylight,Normal,Collision with roadside objects,2,2,Going straight,Driver or rider,Male,31-50,3,Driver,NaN,Not a Pedestrian,Changing lane to the left,Serious Injury
3,1:06:00,Sunday,18-30,Male,Junior high school,Employee,5-10yr,Public (> 45 seats),Governmental,NaN,No defect,Office areas,other,Tangent road with mild grade and flat terrain,Y Shape,Earth roads,Dry,Darkness - lights lit,Normal,Vehicle with vehicle collision,2,2,Going straight,Pedestrian,Female,18-30,3,Driver,Normal,Not a Pedestrian,Changing lane to the right,Slight Injury
4,1:06:00,Sunday,18-30,Male,Junior high school,Employee,2-5yr,NaN,Owner,5-10yrs,No defect,Industrial areas,other,Tangent road with flat terrain,Y Shape,Asphalt roads,Dry,Darkness - lights lit,Normal,Vehicle with vehicle collision,2,2,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury


# **Exploratory Data Analysis**

In [78]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12316 entries, 0 to 12315
Data columns (total 32 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Time                         12316 non-null  object
 1   Day_of_week                  12316 non-null  object
 2   Age_band_of_driver           12316 non-null  object
 3   Sex_of_driver                12316 non-null  object
 4   Educational_level            11575 non-null  object
 5   Vehicle_driver_relation      11737 non-null  object
 6   Driving_experience           11487 non-null  object
 7   Type_of_vehicle              11366 non-null  object
 8   Owner_of_vehicle             11834 non-null  object
 9   Service_year_of_vehicle      8388 non-null   object
 10  Defect_of_vehicle            7889 non-null   object
 11  Area_accident_occured        12077 non-null  object
 12  Lanes_or_Medians             11931 non-null  object
 13  Road_allignment              12

In [79]:
# Distribusi missing value per kolom
missing_pct = (df_raw.isna().sum() / len(df_raw) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].to_string())

Defect_of_vehicle         35.9451
Service_year_of_vehicle   31.8935
Work_of_casuality         25.9662
Fitness_of_casuality      21.3949
Type_of_vehicle            7.7135
Types_of_Junction          7.2020
Driving_experience         6.7311
Educational_level          6.0166
Vehicle_driver_relation    4.7012
Owner_of_vehicle           3.9136
Lanes_or_Medians           3.1260
Vehicle_movement           2.5008
Area_accident_occured      1.9406
Road_surface_type          1.3966
Type_of_collision          1.2585
Road_allignment            1.1530


In [80]:
df_raw.duplicated().sum()

np.int64(0)

In [81]:
df_raw.describe()

,Number_of_vehicles_involved,Number_of_casualties
count,12316.0000,12316.0000
mean,2.0407,1.5481
std,0.6888,1.0072
min,1.0000,1.0000
25%,2.0000,1.0000
50%,2.0000,1.0000
75%,2.0000,2.0000
max,7.0000,8.0000


## **Data Visualization**

In [82]:
vc = df_raw[TARGET_COL].value_counts()

plot_df = vc.reset_index()
plot_df.columns = [TARGET_COL, 'count']
plot_df['label'] = plot_df['count'].apply(lambda x: f'{x:,} ({x/len(df_raw)*100:.1f}%)')

fig = px.bar(plot_df, x=TARGET_COL, y='count', color=TARGET_COL, text='label',
             title='Distribusi Kelas Target: Accident Severity', template='ggplot2')

fig.update_traces(textposition='outside')
fig.update_layout(width=1000, height=700, title={'x': 0.5, 'xanchor': 'center', 'font':{'size':20}}, yaxis_title='Jumlah Sampel', showlegend=False)
fig.show()

In [83]:
fig = px.pie(df_raw,df_raw['Cause_of_accident'],df_raw['Number_of_casualties'],color='Cause_of_accident',template='ggplot2',hole=0.35, title='Analisis Faktor  Penyebab Kecelakaan')
fig.update_layout(width=1200, height=800, title={'x': 0.5, 'xanchor': 'center', 'font':{'size':20}})

In [84]:
fig = px.histogram(df_raw,df_raw['Day_of_week'],df_raw['Number_of_casualties'],color='Day_of_week',template='ggplot2', title='Jumlah Korban Kecelakaan Berdasarkan Hari')
fig.update_layout(width=1200, height=700, title={'x': 0.5, 'xanchor': 'center', 'font':{'size':20}})

In [85]:
fig = px.histogram(df_raw,df_raw['Educational_level'],df_raw['Number_of_casualties'],color='Educational_level',template='ggplot2', title='Kecelakaan Berdasarkan Tingkat Pendidikan')
fig.update_layout(width=1200, height=700, title={'x': 0.5, 'xanchor': 'center', 'font':{'size':20}})

In [86]:
df_raw['Time'] = pd.to_datetime(df_raw['Time'], format='%H:%M:%S')
df_raw['h'] = df_raw['Time'].dt.hour

df_grouped = df_raw.groupby(['h', 'Accident_severity']).size().reset_index(name='counts')
fig = px.line(
    df_grouped, x='h', y='counts',
    color='Accident_severity',
    markers=True,
    template='ggplot2',
    title='Tren Tingkat Keparahan Kecelakaan'
)
fig.update_layout(xaxis_title='Hour of Day', yaxis_title='Number of Accidents', width=1200, height=600, xaxis=dict(dtick=1), title={'x': 0.5, 'xanchor': 'center', 'font':{'size':20}})
fig.show()

# **Feature Engineering**

Fitur dirancang berdasarkan kajian literatur faktor penentu keparahan kecelakaan:
- **Temporal**: jam, periode risiko, hari kerja vs akhir pekan
- **Karakteristik pengemudi**: usia, pengalaman mengemudi
- **Kondisi lingkungan**: permukaan jalan, pencahayaan, cuaca
- **Interaksi antar-fitur**: kondisi ganda, ratio korban-kendaraan
- **Kategorikal penyebab kecelakaan** yang dikelompokkan berdasarkan risiko

> **Catatan**: Kolom `Casualty_severity` tidak digunakan karena merupakan
> turunan langsung dari target dan menyebabkan data leakage.

In [87]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Melakukan seluruh proses rekayasa fitur pada dataframe mentah.

    Tahapan:
    1. Ekstraksi fitur temporal dari kolom Time
    2. Encoding ordinal bermakna sesuai urutan logis domain
    3. Fitur interaksi berbasis riset keparahan kecelakaan
    4. Penghapusan kolom leakage dan redundan
    """
    df = df.copy()

    # ── 5.1 Ekstraksi Fitur Temporal ─────────────────────────────────────────
    df["Hour"] = pd.to_datetime(
        df["Time"], format="%H:%M:%S", errors="coerce"
    ).dt.hour.fillna(-1).astype(int)

    def hour_to_period(h):
        if h in range(6, 9):    return "pagi_sibuk"
        elif h in range(9, 12): return "pagi_normal"
        elif h in range(12, 14): return "siang"
        elif h in range(14, 17): return "sore_normal"
        elif h in range(17, 20): return "sore_sibuk"
        elif h in range(20, 24): return "malam"
        elif h in range(0, 6):  return "dini_hari"
        else:                   return "tidak_diketahui"

    df["Time_period"]  = df["Hour"].apply(hour_to_period)
    df["Is_rush_hour"] = df["Hour"].apply(
        lambda h: 1 if (6 <= h <= 8) or (17 <= h <= 19) else 0
    ).astype(np.int8)
    df["Is_night"] = df["Hour"].apply(
        lambda h: 1 if (h >= 20 or h < 6) else 0
    ).astype(np.int8)

    day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    df["Day_num"]   = df["Day_of_week"].map({d: i for i, d in enumerate(day_order)})
    df["Is_weekend"] = df["Day_num"].apply(lambda x: 1 if x >= 5 else 0).astype(np.int8)

    df = df.drop(columns=["Time"])

    # ── 5.2 Encoding Ordinal Bermakna ─────────────────────────────────────────
    exp_order = {
        "No Licence": 0, "unknown": 0, "Below 1yr": 1,
        "1-2yr": 2, "2-5yr": 3, "5-10yr": 4, "Above 10yr": 5
    }
    df["Driving_exp_num"] = df["Driving_experience"].map(exp_order)

    age_driver_order = {"Under 18": 0, "18-30": 1, "31-50": 2, "Over 51": 3, "Unknown": -1}
    df["Age_driver_num"] = df["Age_band_of_driver"].map(age_driver_order)

    age_cas_order = {"Under 18": 0, "18-30": 1, "31-50": 2, "Over 51": 3,
                     "5": 0, "na": -1}
    df["Age_casualty_num"] = df["Age_band_of_casualty"].map(age_cas_order)

    surface_cond_order = {
        "Dry": 0, "Wet or damp": 1, "Snow": 2, "Flood over 3cm. deep": 3
    }
    df["Road_cond_num"] = df["Road_surface_conditions"].map(surface_cond_order)

    light_order = {
        "Daylight": 0, "Darkness - lights lit": 1,
        "Darkness - lights unlit": 2, "Darkness - no lighting": 3
    }
    df["Light_num"] = df["Light_conditions"].map(light_order)

    service_order = {
        "Below 1yr": 0, "1-2yr": 1, "2-5yrs": 2,
        "5-10yrs": 3, "Above 10yr": 4, "Unknown": -1
    }
    df["Service_year_num"] = df["Service_year_of_vehicle"].map(service_order)

    # ── 5.3 Fitur Interaksi ───────────────────────────────────────────────────
    df["Casualty_vehicle_ratio"] = (
        df["Number_of_casualties"] /
        df["Number_of_vehicles_involved"].clip(lower=1)
    ).astype(np.float32)

    df["Total_impact"] = (
        df["Number_of_casualties"] * df["Number_of_vehicles_involved"]
    ).astype(np.int16)

    weather_risk = {
        "Normal": 0, "Cloudy": 0, "Windy": 1, "Raining": 2,
        "Raining and Windy": 3, "Fog or mist": 3,
        "Snow": 4, "Other": 1, "Unknown": -1
    }
    df["Weather_risk"] = df["Weather_conditions"].map(weather_risk).fillna(-1).astype(int)
    df["Road_weather_interaction"] = (
        df["Road_cond_num"].fillna(0) * df["Weather_risk"].clip(lower=0)
    ).astype(np.int16)

    df["Exp_age_interaction"] = (
        df["Driving_exp_num"].fillna(0) * df["Age_driver_num"].fillna(0)
    ).astype(np.float32)

    has_junction = (~df["Types_of_Junction"].isin(["No junction", np.nan])).astype(int)
    df["High_risk_combo"] = (
        df["Is_night"] & (df["Road_cond_num"].fillna(0) > 0) & has_junction
    ).astype(np.int8)

    cause_risk = {
        "Overspeed": "kecepatan_tinggi", "Driving at high speed": "kecepatan_tinggi",
        "Drunk driving": "pengemudi_mabuk",
        "Driving under the influence of drugs": "pengemudi_mabuk",
        "No priority to vehicle": "pelanggaran",
        "No priority to pedestrian": "pelanggaran",
        "Overtaking": "manuver_berbahaya",
        "Changing lane to the left": "manuver_berbahaya",
        "Changing lane to the right": "manuver_berbahaya",
        "Overloading": "muatan_berlebih",
        "Overturning": "kontrol_hilang", "Turnover": "kontrol_hilang",
        "Driving carelessly": "ceroboh", "Driving to the left": "ceroboh",
        "Moving Backward": "manuver_berbahaya",
        "No distancing": "pelanggaran",
        "Improper parking": "parkir_sembarangan",
        "Getting off the vehicle improperly": "lainnya",
        "Other": "lainnya", "Unknown": "tidak_diketahui"
    }
    df["Cause_category"]      = df["Cause_of_accident"].map(cause_risk).fillna("lainnya")
    df["Is_speed_or_alcohol"] = df["Cause_category"].isin(
        ["kecepatan_tinggi", "pengemudi_mabuk"]
    ).astype(np.int8)

    road_risk = {
        "Asphalt roads": 0, "Asphalt roads with some distress": 1,
        "Gravel roads": 2, "Earth roads": 3, "Other": 2
    }
    df["Road_type_risk"] = df["Road_surface_type"].map(road_risk).fillna(1).astype(int)

    heavy_vehicles = [
        "Lorry (41?100Q)", "Lorry (11?40Q)", "Long lorry",
        "Public (> 45 seats)", "Public (13?45 seats)"
    ]
    df["Is_heavy_vehicle"] = df["Type_of_vehicle"].isin(heavy_vehicles).astype(np.int8)

    # ── 5.4 Hapus Kolom Leakage dan Redundan ─────────────────────────────────
    cols_to_drop = [
        "Casualty_severity",      # Leakage langsung dari target
        "Day_of_week",            # Sudah diekstrak ke Day_num dan Is_weekend
        "Driving_experience",     # Sudah diekstrak ke Driving_exp_num
        "Age_band_of_driver",     # Sudah diekstrak ke Age_driver_num
        "Age_band_of_casualty",   # Sudah diekstrak ke Age_casualty_num
        "Road_surface_conditions","Light_conditions","Service_year_of_vehicle",
        "Cause_of_accident",      # Sudah digrupkan ke Cause_category
        "Weather_conditions",     # Sudah dihitung ke Weather_risk
        "Type_of_vehicle",        # Sudah diproses ke Is_heavy_vehicle
        "Defect_of_vehicle",      # >30% missing, relevansi rendah
    ]
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=cols_to_drop)
    return df

In [88]:
df = engineer_features(df_raw)
print(f"Fitur setelah rekayasa: {df.shape[1]-1} fitur")
print(f"Daftar: {[c for c in df.columns if c != TARGET_COL]}")

Fitur setelah rekayasa: 41 fitur
Daftar: ['Sex_of_driver', 'Educational_level', 'Vehicle_driver_relation', 'Owner_of_vehicle', 'Area_accident_occured', 'Lanes_or_Medians', 'Road_allignment', 'Types_of_Junction', 'Road_surface_type', 'Type_of_collision', 'Number_of_vehicles_involved', 'Number_of_casualties', 'Vehicle_movement', 'Casualty_class', 'Sex_of_casualty', 'Work_of_casuality', 'Fitness_of_casuality', 'Pedestrian_movement', 'h', 'Hour', 'Time_period', 'Is_rush_hour', 'Is_night', 'Day_num', 'Is_weekend', 'Driving_exp_num', 'Age_driver_num', 'Age_casualty_num', 'Road_cond_num', 'Light_num', 'Service_year_num', 'Casualty_vehicle_ratio', 'Total_impact', 'Weather_risk', 'Road_weather_interaction', 'Exp_age_interaction', 'High_risk_combo', 'Cause_category', 'Is_speed_or_alcohol', 'Road_type_risk', 'Is_heavy_vehicle']


# **Data Preprocessing**



In [89]:
def identify_column_types(df: pd.DataFrame, target: str):
    """Identifikasi kolom kategorikal dan numerikal secara otomatis."""
    cat_cols = [c for c in df.select_dtypes(include="object").columns if c != target]
    num_cols = [c for c in df.select_dtypes(exclude="object").columns if c != target]
    return cat_cols, num_cols

In [90]:
def preprocess_pipeline(X_train: pd.DataFrame, X_other: pd.DataFrame, cat_cols: list, num_cols: list, n_neighbors: int = 5):
    """
    Pipeline preprocessing: fit pada train, transform pada data lain.

    Tahapan:
    - Imputasi modus untuk kolom kategorikal
    - Ordinal encoding untuk kolom kategorikal
    - KNN Imputer untuk kolom numerikal
    - Casting ke float32 untuk efisiensi memori
    """
    X_tr = X_train.copy()
    X_ot = X_other.copy()

    for col in num_cols:
        X_tr[col] = pd.to_numeric(X_tr[col], errors="coerce")
        X_ot[col] = pd.to_numeric(X_ot[col], errors="coerce")
    for col in cat_cols:
        X_tr[col] = X_tr[col].astype(object)
        X_ot[col] = X_ot[col].astype(object)

    cat_imputer = SimpleImputer(strategy="most_frequent")
    X_tr[cat_cols] = cat_imputer.fit_transform(X_tr[cat_cols])
    X_ot[cat_cols] = cat_imputer.transform(X_ot[cat_cols])

    oe = OrdinalEncoder(
        handle_unknown="use_encoded_value", unknown_value=-1, dtype=np.float32
    )
    X_tr[cat_cols] = oe.fit_transform(X_tr[cat_cols])
    X_ot[cat_cols] = oe.transform(X_ot[cat_cols])

    num_imputer = KNNImputer(n_neighbors=n_neighbors, weights="distance")
    X_tr[num_cols] = num_imputer.fit_transform(X_tr[num_cols])
    X_ot[num_cols] = num_imputer.transform(X_ot[num_cols])

    all_cols = cat_cols + num_cols
    X_tr[all_cols] = X_tr[all_cols].astype(np.float32)
    X_ot[all_cols] = X_ot[all_cols].astype(np.float32)

    info = {
        "ordinal_encoder": oe, "cat_imputer": cat_imputer,
        "num_imputer": num_imputer, "cat_cols": cat_cols,
        "num_cols": num_cols, "all_feature_names": all_cols,
    }
    return X_tr, X_ot, info

In [91]:
le = LabelEncoder()
y  = le.fit_transform(df[TARGET_COL])
X  = df.drop(columns=[TARGET_COL])
CLASS_NAMES = list(le.classes_)
N_CLASSES   = len(CLASS_NAMES)

print(f"Kelas target : {CLASS_NAMES}")
print(f"Kode integer : {list(range(N_CLASSES))}")

Kelas target : ['Fatal injury', 'Serious Injury', 'Slight Injury']
Kode integer : [0, 1, 2]


In [92]:
X_train_raw, X_test_raw, y_train_all, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
X_tr_raw, X_val_raw, y_train, y_val = train_test_split(
    X_train_raw, y_train_all,
    test_size=VAL_SIZE, random_state=RANDOM_SEED, stratify=y_train_all
)

cat_cols, num_cols = identify_column_types(X_tr_raw, "dummy")

print(f"\nKolom kategorikal : {len(cat_cols)}")
print(f"Kolom numerikal   : {len(num_cols)}")
print(f"\nUkuran split:")
print(f"  Train : {len(X_tr_raw):,} sampel")
print(f"  Val   : {len(X_val_raw):,} sampel")
print(f"  Test  : {len(X_test_raw):,} sampel")


Kolom kategorikal : 18
Kolom numerikal   : 23

Ukuran split:
  Train : 8,897 sampel
  Val   : 1,571 sampel
  Test  : 1,848 sampel


In [93]:
X_train, X_val, enc_info = preprocess_pipeline(X_tr_raw, X_val_raw, cat_cols, num_cols)
_, X_test, _             = preprocess_pipeline(X_tr_raw, X_test_raw, cat_cols, num_cols)

FEATURE_NAMES = enc_info["all_feature_names"]
N_FEATURES    = len(FEATURE_NAMES)
print(f"\nTotal fitur setelah preprocessing: {N_FEATURES}")


Total fitur setelah preprocessing: 41


In [94]:
class_weights_arr  = compute_class_weight(
    "balanced", classes=np.unique(y_train), y=y_train
)
CLASS_WEIGHT_DICT  = {i: w for i, w in enumerate(class_weights_arr)}
SAMPLE_WEIGHT_TRAIN = np.array([CLASS_WEIGHT_DICT[c] for c in y_train])

print(f"\nBobot kelas (imbalance handling):")
for cls, w in zip(CLASS_NAMES, class_weights_arr):
    print(f"  {cls:<20s}: {w:.4f}")


Bobot kelas (imbalance handling):
  Fatal injury        : 26.0146
  Serious Injury      : 2.3537
  Slight Injury       : 0.3942


# **Evolutionary Boosting**

### Konsep Arsitektur

Evolutionary Boosting yang diimplementasikan di sini berbeda dari sekadar
hyperparameter tuning. CatCMAwM mengoptimasi **genome** yang merepresentasikan
struktur lengkap ensemble:

```
Genome (per weak learner k):
  Segment A — x (kontinu) : [weight_k, lr_k, subsample_k, colsample_k]
  Segment B — z (integer)  : [max_depth_k, min_child_weight_k]
  Segment C — c (kategorik): [fitur_1_aktif_k, ..., fitur_N_aktif_k]

Prediksi ensemble:
  P(y|X) = softmax( Σ_k weight_k * P_k(y|X_k) )
  dimana X_k = X[:, feature_mask_k]
```

**Fitness** = Macro F1 pada validation set − λ × penalti kompleksitas

In [99]:
# Ensemble Evolutionary Boosting

class EvoBoostingEnsemble:
    """
    Ensemble Evolutionary Boosting berbasis K weak learner XGBoost.

    Setiap weak learner memiliki konfigurasi independen yang dioptimasi
    oleh CatCMAwM: bobot ensembel, subset fitur, dan hyperparameter pohon.

    Prediksi dilakukan via weighted sum probabilitas setiap weak learner,
    kemudian argmax untuk mendapat label akhir.
    """

    def __init__(self, k: int, n_features: int,
                 n_classes: int, n_estimators: int = N_ESTIMATORS_WL,
                 seed: int = RANDOM_SEED):
        self.k            = k
        self.n_features   = n_features
        self.n_classes    = n_classes
        self.n_estimators = n_estimators
        self.seed         = seed
        self.learners_    = []    # objek model XGBoost setelah fit
        self.feat_masks_  = []    # feature mask per weak learner
        self.weights_     = []    # bobot ensembel per weak learner

    def _decode_genome(self, sol) -> list:
        """
        Mengurai solusi CatCMAwM menjadi spesifikasi K weak learner.

        Format per weak learner:
          x[k*4 : k*4+4] = [weight, lr, subsample, colsample]
          z[k*2 : k*2+2] = [max_depth, min_child_weight]
          c[k*N : k*N+N] = feature_mask (baris ke-i, kolom 0 = True = aktif)
        """
        specs = []
        for k in range(self.k):
            x_off = k * 4
            z_off = k * 2
            c_off = k * self.n_features

            weight           = float(np.clip(sol.x[x_off + 0], 0.01, 1.0))
            learning_rate    = float(np.clip(sol.x[x_off + 1], 0.01, 0.30))
            subsample        = float(np.clip(sol.x[x_off + 2], 0.50, 1.00))
            colsample_bytree = float(np.clip(sol.x[x_off + 3], 0.50, 1.00))
            max_depth        = int(np.clip(int(sol.z[z_off + 0]), 2, 8))
            min_child_weight = int(np.clip(int(sol.z[z_off + 1]), 1, 20))

            feat_mask = sol.c[c_off : c_off + self.n_features, 0].astype(bool)

            if feat_mask.sum() < 5:
                top_idx = np.argsort(
                    sol.c[c_off : c_off + self.n_features, 0].astype(float)
                )[-5:]
                feat_mask = np.zeros(self.n_features, dtype=bool)
                feat_mask[top_idx] = True

            specs.append({
                "weight"          : weight,
                "learning_rate"   : learning_rate,
                "subsample"       : subsample,
                "colsample_bytree": colsample_bytree,
                "max_depth"       : max_depth,
                "min_child_weight": min_child_weight,
                "feat_mask"       : feat_mask,
            })
        return specs

    def fit_from_genome(self, sol, X_tr: pd.DataFrame,
                         y_tr: np.ndarray,
                         sample_weight: np.ndarray = None):
        """Melatih semua weak learner sesuai spesifikasi genome."""
        specs = self._decode_genome(sol)
        self.learners_   = []
        self.feat_masks_ = []
        self.weights_    = []

        for spec in specs:
            selected_cols = [
                FEATURE_NAMES[i]
                for i in range(self.n_features) if spec["feat_mask"][i]
            ]
            model = xgb.XGBClassifier(
                objective="multi:softprob",
                num_class=self.n_classes,
                eval_metric="mlogloss",
                n_estimators=self.n_estimators,
                learning_rate=spec["learning_rate"],
                max_depth=spec["max_depth"],
                min_child_weight=spec["min_child_weight"],
                subsample=spec["subsample"],
                colsample_bytree=spec["colsample_bytree"],
                random_state=self.seed,
                verbosity=0,
                use_label_encoder=False,
            )
            if sample_weight is not None:
                model.fit(X_tr[selected_cols], y_tr, sample_weight=sample_weight)
            else:
                model.fit(X_tr[selected_cols], y_tr)

            self.learners_.append(model)
            self.feat_masks_.append(spec["feat_mask"])
            self.weights_.append(spec["weight"])

        total_w = sum(self.weights_)
        self.weights_ = [w / total_w for w in self.weights_]

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        """Menghitung probabilitas kelas sebagai weighted average dari semua weak learner."""
        agg = np.zeros((len(X), self.n_classes))
        for model, mask, w in zip(self.learners_, self.feat_masks_, self.weights_):
            selected_cols = [FEATURE_NAMES[i] for i in range(self.n_features) if mask[i]]
            prob = model.predict_proba(X[selected_cols])
            agg += w * prob
        return agg

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        """Prediksi label kelas (argmax dari weighted probabilitas)."""
        return np.argmax(self.predict_proba(X), axis=1)

    def get_feature_usage(self) -> pd.DataFrame:
        """Merangkum seberapa sering tiap fitur digunakan lintas weak learner."""
        usage = np.zeros(self.n_features, dtype=int)
        for mask in self.feat_masks_:
            usage += mask.astype(int)
        return pd.DataFrame({
            "Fitur"         : FEATURE_NAMES,
            "Jumlah_WL"     : usage,
            "Pct_Digunakan" : usage / self.k * 100,
        }).sort_values("Jumlah_WL", ascending=False)

In [100]:
# Definisi Search Space CatCMAwM & Fungsi Objektif

def build_catcmawm_optimizer(k: int, n_features: int,
                              seed: int = RANDOM_SEED) -> CatCMAwM:
    """Membangun optimizer CatCMAwM untuk evolutionary boosting."""
    x_space_per = [
        [0.1, 1.0],    # weight ensembel
        [0.01, 0.30],  # learning rate
        [0.50, 1.00],  # subsample
        [0.50, 1.00],  # colsample_bytree
    ]
    z_space_per = [
        list(range(2, 9)),     # max_depth: 2 s.d. 8
        list(range(1, 21)),    # min_child_weight: 1 s.d. 20
    ]
    c_space_per = [2] * n_features

    x_space = x_space_per * k
    z_space = z_space_per * k
    c_space = c_space_per * k

    n_x = len(x_space)
    n_z = len(z_space)
    n_c_vars = len(c_space)
    max_cats  = max(c_space) if c_space else 2

    cat_param = np.zeros((n_c_vars, max_cats))
    cat_param[:, 0] = 0.30
    cat_param[:, 1] = 0.70

    x_means = []
    for lo, hi in x_space_per * k:
        x_means.append((lo + hi) / 2.0)

    z_means = []
    for vals in z_space_per * k:
        z_means.append(float(vals[len(vals) // 2]))

    init_mean = np.array(x_means + z_means, dtype=float)

    optimizer = CatCMAwM(
        x_space=x_space,
        z_space=z_space,
        c_space=c_space,
        mean=init_mean,
        sigma=0.5,
        cat_param=cat_param,
        seed=seed,
    )
    print(f"  CatCMAwM dibangun:")
    print(f"    x (kontinu) : {len(x_space)} variabel ({len(x_space_per)} per WL × {k} WL)")
    print(f"    z (integer)  : {len(z_space)} variabel ({len(z_space_per)} per WL × {k} WL)")
    print(f"    c (kategorik): {len(c_space)} variabel ({n_features} fitur per WL × {k} WL)")
    print(f"    Population size: {optimizer.population_size}")
    return optimizer


def compute_complexity_penalty(sol, k: int, n_features: int) -> float:
    """Menghitung penalti kompleksitas genome untuk mencegah overfitting ensembel."""
    total_active_feats = 0
    total_depth        = 0
    for k_idx in range(k):
        c_off = k_idx * n_features
        feat_mask = sol.c[c_off : c_off + n_features, 0].astype(bool)
        total_active_feats += feat_mask.sum()
        z_off = k_idx * 2
        total_depth += int(sol.z[z_off])
    norm_feat  = total_active_feats / (k * n_features)
    norm_depth = total_depth / (k * 8)
    return 0.5 * norm_feat + 0.5 * norm_depth


def evaluate_genome_fitness(sol,
                             k: int,
                             n_features: int,
                             n_classes: int,
                             X_tr: pd.DataFrame,
                             y_tr: np.ndarray,
                             X_v: pd.DataFrame,
                             y_v: np.ndarray,
                             sample_weight: np.ndarray = None,
                             n_estimators: int = N_ESTIMATORS_WL,
                             seed: int = RANDOM_SEED) -> float:
    """
    Fungsi objektif untuk CatCMAwM: negatif fitness genome ensembel.
    Fitness = Macro F1 pada validation set − λ × penalti kompleksitas
    """
    try:
        agg_prob = np.zeros((len(X_v), n_classes))
        weights  = []
        for k_idx in range(k):
            x_off = k_idx * 4
            z_off = k_idx * 2
            c_off = k_idx * n_features

            weight           = float(np.clip(sol.x[x_off + 0], 0.01, 1.0))
            learning_rate    = float(np.clip(sol.x[x_off + 1], 0.01, 0.30))
            subsample        = float(np.clip(sol.x[x_off + 2], 0.50, 1.00))
            colsample_bytree = float(np.clip(sol.x[x_off + 3], 0.50, 1.00))
            max_depth        = int(np.clip(int(sol.z[z_off + 0]), 2, 8))
            min_child_weight = int(np.clip(int(sol.z[z_off + 1]), 1, 20))

            feat_mask = sol.c[c_off : c_off + n_features, 0].astype(bool)
            if feat_mask.sum() < 5:
                top_idx   = np.argsort(
                    sol.c[c_off : c_off + n_features, 0].astype(float)
                )[-5:]
                feat_mask = np.zeros(n_features, dtype=bool)
                feat_mask[top_idx] = True

            selected_cols = [FEATURE_NAMES[i] for i in range(n_features) if feat_mask[i]]

            model = xgb.XGBClassifier(
                objective="multi:softprob",
                num_class=n_classes,
                eval_metric="mlogloss",
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                max_depth=max_depth,
                min_child_weight=min_child_weight,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                random_state=seed,
                verbosity=0,
                use_label_encoder=False,
            )
            if sample_weight is not None:
                model.fit(X_tr[selected_cols], y_tr, sample_weight=sample_weight)
            else:
                model.fit(X_tr[selected_cols], y_tr)

            prob = model.predict_proba(X_v[selected_cols])
            agg_prob += weight * prob
            weights.append(weight)

        total_w  = sum(weights)
        agg_prob = agg_prob / total_w
        y_pred = np.argmax(agg_prob, axis=1)
        macro_f1 = f1_score(y_v, y_pred, average="macro", zero_division=0)
        penalty  = compute_complexity_penalty(sol, k, n_features)
        fitness  = macro_f1 - LAMBDA_PENALTY * penalty
        return -fitness
    except Exception as e:
        return 0.0


def run_evolutionary_boosting(X_tr, y_tr, X_v, y_v,
                               k=K_WEAK_LEARNERS,
                               max_iter=EVO_MAX_ITER,
                               seed=RANDOM_SEED,
                               verbose_every=5):
    """Menjalankan Evolutionary Boosting menggunakan CatCMAwM."""
    optimizer = build_catcmawm_optimizer(k, N_FEATURES, seed)
    sw = SAMPLE_WEIGHT_TRAIN if (
        len(y_tr) == len(SAMPLE_WEIGHT_TRAIN)
    ) else np.array([CLASS_WEIGHT_DICT[c] for c in y_tr])

    best_score = np.inf
    best_sol   = None
    history    = []

    print(f"\n  ┌─ Evolutionary Boosting (CatCMAwM) mulai")
    print(f"  │  K={k} weak learners, maks {max_iter} generasi, pop={optimizer.population_size}")
    t0 = time.time()

    for gen in range(max_iter):
        solutions = []
        for _ in range(optimizer.population_size):
            sol   = optimizer.ask()
            score = evaluate_genome_fitness(
                sol, k, N_FEATURES, N_CLASSES, X_tr, y_tr, X_v, y_v, sample_weight=sw
            )
            solutions.append((sol, score))

        optimizer.tell(solutions)
        gen_scores = [s[1] for s in solutions]
        gen_best   = min(gen_scores)
        history.append(-gen_best)

        if gen_best < best_score:
            best_score = gen_best
            best_sol   = solutions[int(np.argmin(gen_scores))][0]

        if (gen + 1) % verbose_every == 0:
            elapsed = time.time() - t0
            print(f"  │  Gen {gen+1:3d}  best_F1={-best_score:.4f}  "
                  f"gen_F1={-gen_best:.4f}  elapsed={elapsed:.1f}s")

        if optimizer.should_stop():
            print(f"  │  Konvergensi tercapai pada generasi {gen+1}")
            break

    elapsed = time.time() - t0
    print(f"  └─ Selesai. Best Macro F1 = {-best_score:.4f} | Waktu: {elapsed:.1f}s")
    return best_sol, -best_score, history

# **Training XGBoost Baseline**

Model XGBoost tunggal dengan hyperparameter default dilatih pada data train.
Evaluasi dilakukan pada **validation set** — bukan test set.

In [ ]:
val_results    = {}   # Hasil evaluasi fase 1 & 2 pada validation set
test_results   = {}   # Hasil evaluasi fase 4 pada test set
trained_models = {}   # Penyimpanan objek model

MODEL_COLORS = {
    "XGBoost Baseline"       : "#1565C0",
    "Evolutionary Boosting"  : "#E53935",
}

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Menghitung semua metrik evaluasi klasifikasi multiclass."""
    return {
        "Accuracy" : accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall"   : recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1"       : f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_pred"   : y_pred,
    }

In [101]:
xgb_baseline = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=N_CLASSES,
    eval_metric="mlogloss",
    random_state=RANDOM_SEED,
    verbosity=0,
    use_label_encoder=False,
)
xgb_baseline.fit(X_train, y_train, sample_weight=SAMPLE_WEIGHT_TRAIN)
trained_models["XGBoost Baseline"] = xgb_baseline

y_pred_baseline = xgb_baseline.predict(X_val)
val_results["XGBoost Baseline"] = compute_metrics(y_val, y_pred_baseline)
m = val_results["XGBoost Baseline"]

print(f"\n  ✓ XGBoost Baseline")
print(f"    Accuracy  : {m['Accuracy']:.4f}")
print(f"    Macro F1  : {m['F1']:.4f}")
print(f"    Precision : {m['Precision']:.4f}")
print(f"    Recall    : {m['Recall']:.4f}")


  ✓ XGBoost Baseline
    Accuracy  : 0.7619
    Macro F1  : 0.5108
    Precision : 0.5735
    Recall    : 0.4838


# **Evolutionary Boosting (CatCMAwM)**

CatCMAwM mengoptimasi genome yang merepresentasikan struktur lengkap ensemble:
- **Segment A (kontinu)**: bobot ensembel, learning rate, subsample, colsample per WL
- **Segment B (integer)**: max_depth, min_child_weight per WL
- **Segment C (kategorik)**: feature mask biner (aktif/tidak) per fitur per WL

In [102]:
best_sol, best_evo_f1, evo_history = run_evolutionary_boosting(
    X_train, y_train, X_val, y_val,
    k=K_WEAK_LEARNERS,
    max_iter=EVO_MAX_ITER,
    seed=RANDOM_SEED,
)

evo_ensemble = EvoBoostingEnsemble(
    k=K_WEAK_LEARNERS,
    n_features=N_FEATURES,
    n_classes=N_CLASSES,
    n_estimators=N_ESTIMATORS_WL,
    seed=RANDOM_SEED,
)
evo_ensemble.fit_from_genome(best_sol, X_train, y_train,
                              sample_weight=SAMPLE_WEIGHT_TRAIN)
trained_models["Evolutionary Boosting"] = evo_ensemble

y_pred_evo = evo_ensemble.predict(X_val)
val_results["Evolutionary Boosting"] = compute_metrics(y_val, y_pred_evo)
m = val_results["Evolutionary Boosting"]

print(f"\n  ✓ Evolutionary Boosting (Genome terbaik dilatih ulang pada Train)")
print(f"    Accuracy  : {m['Accuracy']:.4f}")
print(f"    Macro F1  : {m['F1']:.4f}")
print(f"    Precision : {m['Precision']:.4f}")
print(f"    Recall    : {m['Recall']:.4f}")

  CatCMAwM dibangun:
    x (kontinu) : 20 variabel (4 per WL × 5 WL)
    z (integer)  : 10 variabel (2 per WL × 5 WL)
    c (kategorik): 205 variabel (41 fitur per WL × 5 WL)
    Population size: 20

  ┌─ Evolutionary Boosting (CatCMAwM) mulai
  │  K=5 weak learners, maks 50 generasi, pop=20
  │  Gen   5  best_F1=0.5296  gen_F1=0.5261  elapsed=397.8s
  │  Gen  10  best_F1=0.5296  gen_F1=0.5253  elapsed=789.8s
  │  Gen  15  best_F1=0.5398  gen_F1=0.5344  elapsed=1200.6s
  │  Gen  20  best_F1=0.5572  gen_F1=0.5559  elapsed=1620.8s
  │  Gen  25  best_F1=0.5834  gen_F1=0.5700  elapsed=2048.1s
  │  Gen  30  best_F1=0.5834  gen_F1=0.5823  elapsed=2493.2s
  │  Gen  35  best_F1=0.5882  gen_F1=0.5821  elapsed=2945.7s
  │  Gen  40  best_F1=0.5882  gen_F1=0.5811  elapsed=3396.5s
  │  Gen  45  best_F1=0.6131  gen_F1=0.6131  elapsed=3831.3s
  │  Gen  50  best_F1=0.6322  gen_F1=0.6064  elapsed=4287.3s
  └─ Selesai. Best Macro F1 = 0.6322 | Waktu: 4287.3s

  ✓ Evolutionary Boosting (Genome terbaik di

# **Kurva Konvergensi Evolutionary Boosting**

In [130]:
fig = px.line(
    x=range(1, len(evo_history)+1), y=evo_history, markers=True,
    labels={"x":"Generasi","y":"Best Macro F1 (fitness)"},
    template="ggplot2", title="Kurva Konvergensi Evolutionary Boosting (CatCMAwM)"
)
fig.update_traces(line_color="#E53935", line_width=2, marker_size=4,
                  name="Best Macro F1 (fitness)")
fig.add_scatter(x=[1, len(evo_history)], y=[evo_history[-1]]*2,
                mode="lines", line_dash="dash",
                name=f"Nilai akhir: {evo_history[-1]:.4f}")
fig.update_layout(
    width=1000, height=600,
    legend=dict(x=.65, y=.98, bgcolor="rgba(255,255,255,.8)")
)
fig.show()

# **Evaluation**

In [174]:
def plot_comparison_bars(results_store):
    names  = list(results_store)
    colors = [MODEL_COLORS.get(n, "#90A4AE") for n in names]
    accs   = [results_store[n]["Accuracy"] for n in names]
    f1s    = [results_store[n]["F1"] for n in names]
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f"Perbandingan Accuracy",
                        f"Perbandingan Macro F1"]
    )
    for col, vals, ylabel in [(1, accs, "Accuracy"), (2, f1s, "Macro F1-Score")]:
        fig.add_bar(x=names, y=vals, marker_color=colors,
                    text=[f"{v:.4f}" for v in vals], textposition="outside",
                    row=1, col=col)
        fig.update_yaxes(title_text=ylabel, range=[0, min(1, max(vals)+0.12)],
                         row=1, col=col)
    fig.update_layout(template="ggplot2", width=1400, height=500, showlegend=False)
    fig.show()

In [143]:
plot_comparison_bars(val_results)

In [176]:
fig = make_subplots(rows=1, cols=2, subplot_titles=list(val_results), horizontal_spacing=0.15)
for i, (_, m) in enumerate(val_results.items(), 1):
    cm = confusion_matrix(y_val, m["y_pred"])
    fig.add_trace(
        go.Heatmap(z=cm, x=CLASS_NAMES, y=CLASS_NAMES,
                   colorscale="Blues", text=cm, texttemplate="%{text}",
                   showscale=False, xgap=1, ygap=1),
        row=1, col=i
    )

fig.update_layout(template="ggplot2", width=1200, height=600)
fig.update_xaxes(title="Prediksi", tickangle=30)
fig.update_yaxes(title="Aktual", autorange="reversed")
fig.show()

In [106]:
baseline_f1 = val_results["XGBoost Baseline"]["F1"]
evo_f1      = val_results["Evolutionary Boosting"]["F1"]
delta_f1    = evo_f1 - baseline_f1
delta_acc   = (val_results["Evolutionary Boosting"]["Accuracy"]
               - val_results["XGBoost Baseline"]["Accuracy"])

print("\n" + "=" * 65)
print("RINGKASAN PERBANDINGAN — Validation Set")
print("=" * 65)
print(f"\n  {'Model':<28s}  {'Accuracy':>9s}  {'Macro F1':>9s}")
print(f"  {'-'*28}  {'-'*9}  {'-'*9}")
for name, m in val_results.items():
    mark = " ← terbaik" if m["F1"] == max(r["F1"] for r in val_results.values()) else ""
    print(f"  {name:<28s}  {m['Accuracy']:>9.4f}  {m['F1']:>9.4f}{mark}")

print(f"\n  Delta Accuracy (Evo − Baseline) : {delta_acc:+.4f}")
print(f"  Delta Macro F1 (Evo − Baseline) : {delta_f1:+.4f}")

if delta_f1 > 0:
    print("\n  ✅ Evolutionary Boosting unggul dibandingkan XGBoost Baseline.")
elif delta_f1 == 0:
    print("\n  ⚠️  Performa setara. Coba naikkan EVO_MAX_ITER.")
else:
    print("\n  ⚠️  Baseline lebih baik. Coba naikkan EVO_MAX_ITER atau K_WEAK_LEARNERS.")


RINGKASAN PERBANDINGAN — Validation Set

  Model                          Accuracy   Macro F1
  ----------------------------  ---------  ---------
  XGBoost Baseline                 0.7619     0.5108
  Evolutionary Boosting            0.7976     0.6593 ← terbaik

  Delta Accuracy (Evo − Baseline) : +0.0356
  Delta Macro F1 (Evo − Baseline) : +0.1484

  ✅ Evolutionary Boosting unggul dibandingkan XGBoost Baseline.


In [107]:
print("\n  Detail Konfigurasi Ensemble Evolutionary Boosting:")
print(f"  Jumlah weak learner : {evo_ensemble.k}")
print(f"  Bobot ternormalisasi: {[round(w,3) for w in evo_ensemble.weights_]}")
for i, (mask, w) in enumerate(zip(evo_ensemble.feat_masks_, evo_ensemble.weights_)):
    selected = [FEATURE_NAMES[j] for j in range(N_FEATURES) if mask[j]]
    print(f"\n  Weak Learner {i+1} (bobot={w:.3f}):")
    print(f"    Fitur aktif ({mask.sum()}/{N_FEATURES}): {selected}")


  Detail Konfigurasi Ensemble Evolutionary Boosting:
  Jumlah weak learner : 5
  Bobot ternormalisasi: [0.289, 0.126, 0.155, 0.161, 0.269]

  Weak Learner 1 (bobot=0.289):
    Fitur aktif (13/41): ['Area_accident_occured', 'Types_of_Junction', 'Cause_category', 'Number_of_casualties', 'h', 'Is_night', 'Day_num', 'Is_weekend', 'Road_cond_num', 'Light_num', 'Casualty_vehicle_ratio', 'Total_impact', 'High_risk_combo']

  Weak Learner 2 (bobot=0.126):
    Fitur aktif (13/41): ['Educational_level', 'Area_accident_occured', 'Road_allignment', 'Types_of_Junction', 'Number_of_vehicles_involved', 'Is_rush_hour', 'Is_night', 'Day_num', 'Is_weekend', 'Age_driver_num', 'Road_cond_num', 'Light_num', 'High_risk_combo']

  Weak Learner 3 (bobot=0.155):
    Fitur aktif (18/41): ['Sex_of_driver', 'Types_of_Junction', 'Road_surface_type', 'Vehicle_movement', 'Work_of_casuality', 'Fitness_of_casuality', 'Time_period', 'Number_of_vehicles_involved', 'Number_of_casualties', 'h', 'Hour', 'Is_night', 'Is_we

In [158]:
y_test_bin  = label_binarize(y_val, classes=list(range(N_CLASSES)))
y_prob_test = y_pred_evo = evo_ensemble.predict_proba(X_val)

In [173]:
macro_roc_auc = roc_auc_score(
    y_test_bin, y_prob_test, average="macro", multi_class="ovr"
)
print("ROC-AUC per Kelas (Test Set):")
for cls_name, auc_val in roc_aucs.items():
    print(f"  {cls_name:<20s}: {auc_val:.4f}")
print(f"  {'Macro ROC-AUC':<20s}: {macro_roc_auc:.4f}")

ROC-AUC per Kelas (Test Set):
  Fatal injury        : 0.9230
  Serious Injury      : 0.7360
  Slight Injury       : 0.7483
  Macro ROC-AUC       : 0.8024


In [159]:
CLASS_COLORS, CLASS_LINESTYLE = ["#F44336", "#FF9800", "#2196F3"], ["-", "--", "-."]
roc_df, color_map, roc_aucs = [], {}, {}

for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_test[:, i])
    roc_auc = auc(fpr, tpr)
    roc_aucs[cls_name], label = roc_auc, f"{cls_name} (AUC={roc_auc:.4f})"
    color_map[label] = color
    roc_df.append(pd.DataFrame({"False Positive Rate": fpr, "True Positive Rate": tpr, "Class": label}))

roc_df = pd.concat(roc_df, ignore_index=True)

fig_roc = px.line(
    roc_df, x="False Positive Rate", y="True Positive Rate",
    color="Class", color_discrete_map=color_map, template="ggplot2", title="ROC Curve"
)
fig_roc.add_scatter(x=[0, 1], y=[0, 1], mode="lines", name="", line=dict(color="black", dash="dash"))
fig_roc.update_layout(
    width=1000, height=800, title_x=0.5, xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1.05]),
    legend=dict(x=0.98, y=0.02, xanchor="right", yanchor="bottom", bgcolor="rgba(255,255,255,0.8)", bordercolor="black", borderwidth=1)
)
fig_roc.show()

In [182]:
pr_df, color_map = [], {}

for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, CLASS_COLORS)):
    precision, recall, _ = precision_recall_curve(y_test_bin[:, i], y_prob_test[:, i])
    ap = average_precision_score(y_test_bin[:, i], y_prob_test[:, i])

    label = f"{cls_name} (AP={ap:.4f})"
    color_map[label] = color
    pr_df.append(pd.DataFrame({"Recall": recall, "Precision": precision, "Class": label}))

pr_df = pd.concat(pr_df, ignore_index=True)

fig_pr = px.line(
    pr_df, x="Recall", y="Precision", color="Class",
    color_discrete_map=color_map, template="ggplot2", title="Precision-Recall Curve"
)
fig_pr.update_layout(
    width=1000, height=800, title_x=0.5, xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1.05]),
    legend=dict(x=0.28, y=0.02, xanchor="right", yanchor="bottom", bgcolor="rgba(255,255,255,0.8)", bordercolor="black", borderwidth=1)
)
fig_pr.show()

In [183]:
feat_usage = evo_ensemble.get_feature_usage()
feat_usage = feat_usage.sort_values("Jumlah_WL", ascending=False).head(20)

fig = px.bar(
    feat_usage, x="Jumlah_WL", y="Fitur", orientation="h",
    text="Jumlah_WL",
    color=feat_usage["Jumlah_WL"].map(
        lambda x: "#E53935" if x == K_WEAK_LEARNERS else "#FF7043" if x > 0 else "#ECEFF1"
    ),
    title="Top 20 Konsistensi Penggunaan Fitur",
    template="ggplot2"
)
fig.update_traces(textposition="outside")
fig.update_layout(
    width=1000, height=600,
    xaxis_title="Digunakan oleh N weak learner",
    yaxis_title=None, showlegend=False
)
fig.update_xaxes(range=[0, K_WEAK_LEARNERS + .5], tickmode="linear", dtick=1)
fig.update_yaxes(categoryorder="total ascending")
fig.show()